# Difficulty Concept Location Notebook

## CIS 5450: Big Data Analytics — Flight Delay Prediction

**Team**: Xiaoyang Wan · Dong Dong · Yihong Yu · Yanchen Zhou
**Date**: April 28, 2026

This notebook documents where each difficulty concept from the course is applied in our project.
For each concept we list: (1) *what* the concept is, (2) *where* in the project it is used
(notebook & section), and (3) *why* we chose to use it.

本 notebook 列出本项目用到的所有"课程难度概念"的使用位置、方法和目的。

---

## Concept 1 — Large-Scale Data Wrangling with Pandas

**Concept**: Handling multi-gigabyte tabular data with `pandas` (dtype optimization,
parquet format, chunked I/O, `category` dtype).

**Where**:
- `notebooks/01_data_process/BTS_cleaning.ipynb` — reading 12 monthly CSVs (~7M rows) and
  writing a single consolidated `flights_2024_clean.parquet`
- `notebooks/01_data_process/weather_process.ipynb` — parsing ISD-Lite fixed-width format

**Why**: CSV is ~10× slower to read and ~5× larger on disk than parquet. For a 6.8M-row dataset,
parquet + typed columns reduced load time from minutes to seconds across our pipeline.

---

## Concept 2 — Temporal Data Integration via `merge_asof`

**Concept**: Joining irregular-interval time series (weather observations at mostly-but-not-always
hourly cadence) to minute-precise event data (flight schedules). `pd.merge_asof` solves this by
matching each left-side row to the closest right-side timestamp within a tolerance.

**Where**: `notebooks/02_data_integration/weather_join.ipynb` — joining each flight's scheduled
departure time to the nearest-hour weather observation at its origin (and destination) airport.

**Why**: A naive equi-join on `(airport, hour)` drops rows whenever the weather station skipped
an hour of observations (happens for ~5% of hours in 2024). `merge_asof(direction="nearest",
tolerance="1h")` preserves nearly all flight rows while still respecting physical plausibility.

---

## Concept 3 — Feature Engineering Without Data Leakage

**Concept**: Engineered features that incorporate history must use only **past** information.
Naive `rolling(7).mean()` on the target column leaks today's target into today's feature.

**Where**:
- `notebooks/02_data_integration/feature_engineering.ipynb` — all rolling-history features
  (`airline_delay_rate_7d`, `origin_delay_rate_7d`, `route_delay_rate_7d`) use
  **`shift(1)` before the rolling window**
- Cascading feature `prev_flight_arr_delay` looks only at the **prior** flight of the same
  aircraft (by tail number), sorted chronologically

**Why**: Without `shift(1)`, cross-validation scores were inflated by ~5-6 pp AUC —
unreliable model selection.

**Code excerpt** (from `feature_engineering.ipynb`):
```python
df["airline_delay_rate_7d"] = (
    df.sort_values(["Reporting_Airline", "FlightDate"])
      .groupby("Reporting_Airline")["DepDel15"]
      .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
)
```

---

## Concept 4 — Simulation-Based Hypothesis Testing

**Concept**: Instead of relying on parametric test assumptions (normality, equal variance),
we compute p-values by simulation from the null distribution.
Three flavors are used in class:
- **Permutation test** — shuffle group labels B times, recompute statistic
- **Bootstrap** — resample with replacement to construct CIs
- **Monte Carlo** — sample from an explicit null model (e.g., independence)

**Where**: `notebooks/03_eda/hypothesis_testing.ipynb` — four tests:

| Test | Method | Why |
|---|---|---|
| Budget vs. Legacy airlines | Permutation (B=10k) | Null: labels exchangeable |
| Summer vs. Winter | Bootstrap CI (B=10k) | Want a CI, not just p-value |
| Hub vs. Non-Hub | Permutation (B=10k) | Same rationale as test 1 |
| Weather vs. Delay | Monte Carlo χ² (B=10k) | Independence as null; sample from it |

**Why simulation vs. analytic**: Our samples are large (500k), so we don't need MC to get a
p-value — but simulation makes the **assumptions explicit** and works for arbitrary test
statistics. It's also what was taught in class.

---

## Concept 5 — Class Imbalance Handling

**Concept**: When positive-class prevalence is low (here ~20%), naive loss functions
pay more attention to the majority class. Standard mitigations:
- **Class weights** — reweight the loss so minority errors cost more
- **SMOTE** — synthesize new minority-class training points via nearest-neighbor interpolation
- **Threshold tuning** — change the decision threshold from 0.5 to maximize F1

**Where**: `notebooks/04_modeling/advanced_model.ipynb` — compares all three.

- Class weights: `XGBClassifier(scale_pos_weight=N_neg/N_pos)`,
  `LGBMClassifier(is_unbalance=True)`
- SMOTE: `imblearn.SMOTE(sampling_strategy=1.0)`
- Threshold: precision-recall curve grid search for best F1

**Finding**: On our data, class weights beat SMOTE on recall/AUC; SMOTE boosted precision
but sacrificed recall. Threshold tuning gave an additional +0.5 pp F1 on top of tuned weights.

---

## Concept 6 — Logistic Regression at Scale (SGD)

**Concept**: `sklearn.linear_model.LogisticRegression` uses batch solvers (liblinear, lbfgs,
saga) that scale poorly beyond ~1M rows. `SGDClassifier(loss="log_loss")` implements
stochastic gradient descent — per-row updates, constant memory, linear scaling.

**Where**: `notebooks/04_modeling/baseline_model.ipynb`

**Why**: `LogisticRegression(solver="saga")` timed out at 30 min on our 5.6M-row training set.
`SGDClassifier` trained in seconds. For probability output (SGD doesn't expose `predict_proba`
by default with `log_loss`), we apply the sigmoid manually to `decision_function` output.

---

## Concept 7 — Gradient Boosting on Tabular Data

**Concept**: Gradient-boosted decision trees (GBDT) — iteratively fit shallow trees on the
negative gradient of the loss. XGBoost and LightGBM are two leading implementations with
histogram-based splitting for scalability.

**Where**:
- `notebooks/04_modeling/advanced_model.ipynb` — initial XGBoost + LightGBM
- `notebooks/04_modeling/tuning_optimization.ipynb` — `RandomizedSearchCV` with 30 iterations
  × 3-fold CV over hyperparameters
- `notebooks/04_modeling/fulldata_final_model.ipynb` — full 5.6M-row training

**Why**: Consistently strong performers on heterogeneous tabular data. Achieved our best
AUC (0.82), dominating linear and RF baselines.

---

## Concept 8 — Hyperparameter Tuning with Cross-Validation

**Concept**: Sample hyperparameter combinations from a distribution and evaluate each via
K-fold CV. Random search is more efficient than grid search at high dimensionality.

**Where**: `notebooks/04_modeling/tuning_optimization.ipynb`

```python
search = RandomizedSearchCV(
    xgb_base,
    param_distributions={
        "n_estimators":    [100, 200, 300, 400],
        "max_depth":       [4, 5, 6, 8, 10],
        "learning_rate":   [0.01, 0.05, 0.1, 0.2],
        "min_child_weight":[1, 3, 5, 10, 20],
        "subsample":       [0.6, 0.7, 0.8, 0.9],
        "colsample_bytree":[0.6, 0.7, 0.8, 0.9],
        "reg_alpha":       [0, 0.1, 0.5, 1],
        "reg_lambda":      [0.5, 1, 2, 5],
    },
    n_iter=30, cv=StratifiedKFold(3), scoring="roc_auc",
)
```

**Why**: 30 random samples from a 4096-point grid gives ~99% probability of finding a config
within the top 10% — much cheaper than exhaustive grid search while almost as effective.

---

## Concept 9 — Decision Threshold Optimization

**Concept**: Classifier probability thresholds other than 0.5 can maximize F1, Precision, or
any other operating-point metric. Computed via precision-recall curve.

**Where**: `notebooks/04_modeling/tuning_optimization.ipynb`,
`notebooks/CIS5450_final_project.ipynb` § 8.7

```python
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1 = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-10)
best_t = thresholds[np.argmax(f1)]
```

**Why**: Our tuned XGBoost at default `t=0.5` yielded F1=0.582. At `t=0.566` it reaches F1=0.587
— not earth-shattering, but free, and importantly shows how to trade precision for recall
depending on stakeholder preferences.

---

## Concept 10 — Temporal Train/Test Split

**Concept**: For time-indexed data, random train/test split leaks future information into
training. The correct approach is a **temporal split** — train on an earlier time window,
test on a later one.

**Where**: All four modeling notebooks use the same split:
- Train: Jan–Oct 2024
- Test: Nov–Dec 2024

**Why**: Rolling features (`*_delay_rate_7d`) in December implicitly contain information about
November. If we random-split, the model can memorize November → December transitions.
A temporal holdout simulates true forecasting: predict the unseen future from the observed past.

---

## Concept 11 — ROC-AUC and Precision-Recall Analysis

**Concept**: ROC curves (TPR vs FPR) summarize classifier performance across all thresholds;
AUC is the probability that a random positive ranks above a random negative.
On class-imbalanced problems, precision-recall curves complement ROC.

**Where**:
- `notebooks/04_modeling/baseline_model.ipynb` — first ROC curves
- `notebooks/CIS5450_final_project.ipynb` § 8.8 — all 6 models overlaid

**Why**: AUC is threshold-invariant and therefore robust for model comparison. F1 (our
headline number) depends on threshold choice; AUC does not. Showing both tells the complete
story.

---

## Concept 12 — Linear vs. Ridge Regression + Residual Analysis

**Concept**: Ordinary least squares (OLS) minimizes squared residuals; Ridge adds L2
regularization to stabilize coefficients when features are correlated.
Residual plots diagnose heteroskedasticity, nonlinearity, and outliers.

**Where**: `notebooks/04_modeling/regression_model.ipynb`

**Why**:
- Our feature matrix has one-hot-encoded airport columns (hundreds of nearly-orthogonal dummies)
  but also correlated weather features (temperature ↔ dew point) — Ridge stabilizes the fit
- Residual analysis revealed the target is **heavy-tailed** and error grows with predicted value,
  motivating the jump to nonlinear models (LightGBM)

---

## Concept 13 — Data Visualization for EDA

**Concept**: Matplotlib + Seaborn workflows for bar plots, scatter plots, ridgeline plots,
box plots, heatmaps, and geographic overlays.

**Where**: `notebooks/03_eda/eda.ipynb` (~25 charts) +
`notebooks/CIS5450_final_project.ipynb` § 6 (8 headline charts)

**Why**: Patterns like the cascading-delay relationship (§6.4) are much more compelling
as a chart than as a correlation coefficient. Visualization is the bridge from
"the data contains signal X" to "the stakeholder understands signal X."

---

## Concept 14 — Stratified Subsampling for Tractability

**Concept**: Random sampling preserves class distribution in expectation but can drift on
small samples. Stratified sampling takes fixed proportions from each class.

**Where**: `notebooks/CIS5450_final_project.ipynb` § 8.2 — 500k training subsample

```python
pos_idx = train_idx[y_full_train == 1]
neg_idx = train_idx[y_full_train == 0]
ratio = len(pos_idx) / len(train_idx)
sampled = np.concatenate([
    rng.choice(pos_idx, int(500_000 * ratio), replace=False),
    rng.choice(neg_idx, int(500_000 * (1-ratio)), replace=False),
])
```

**Why**: Guarantees the subsample has exactly the same class balance as the full training
set — particularly important for hyperparameter tuning where CV fold balance affects
metric stability. We validated that 500k ≈ full 5.6M performance (Δ AUC < 0.002).

---

## Concept 15 — Feature Importance Interpretation

**Concept**: Tree models provide "gain"-based importance — how much each feature reduces
the loss when used as a split. Useful for feature selection and stakeholder communication.

**Where**: `notebooks/04_modeling/advanced_model.ipynb`,
`notebooks/CIS5450_final_project.ipynb` § 8.8

**Why**: `prev_flight_arr_delay` has dominant importance — this isn't just a statistical
observation, it's an actionable insight: airlines can focus on **aircraft rotation management**
as the single highest-leverage intervention to reduce system-wide delays.

---

## Summary Table

| # | Concept | Primary Notebook | Why It Matters Here |
|---|---|---|---|
| 1 | Large-scale pandas + parquet | `01_data_process/BTS_cleaning.ipynb` | 6.8M rows makes dtype + format critical |
| 2 | `merge_asof` temporal join | `02_data_integration/weather_join.ipynb` | Irregular weather ↔ minute-precise flights |
| 3 | Leakage-free rolling features | `02_data_integration/feature_engineering.ipynb` | `shift(1)` prevents target contamination |
| 4 | Simulation-based hypothesis tests | `03_eda/hypothesis_testing.ipynb` | 4 tests: permutation, bootstrap, Monte Carlo |
| 5 | Class imbalance handling | `04_modeling/advanced_model.ipynb` | SMOTE vs class weights vs threshold |
| 6 | SGD-based Logistic Regression | `04_modeling/baseline_model.ipynb` | `LogisticRegression(saga)` timed out |
| 7 | XGBoost / LightGBM | `04_modeling/advanced_model.ipynb` | Best AUC on tabular data |
| 8 | `RandomizedSearchCV` + K-fold CV | `04_modeling/tuning_optimization.ipynb` | 30-iter random search, 3-fold CV |
| 9 | PR-curve threshold tuning | `04_modeling/tuning_optimization.ipynb` | Free F1 gains at optimal threshold |
| 10 | Temporal train/test split | All modeling notebooks | Jan–Oct train, Nov–Dec test |
| 11 | ROC-AUC + PR curves | `04_modeling/*.ipynb` | Threshold-invariant model comparison |
| 12 | Linear/Ridge + residual analysis | `04_modeling/regression_model.ipynb` | Target is heavy-tailed heteroskedastic |
| 13 | EDA visualization | `03_eda/eda.ipynb` | ~25 charts; cascading effect is the headline |
| 14 | Stratified subsampling | `CIS5450_final_project.ipynb` | Preserves class balance at 500k |
| 15 | Feature importance | All modeling notebooks | `prev_flight_arr_delay` dominance |